# Titler v2 — 8+4 / d320 / gated-gelu (Colab free T4)

Thin driver around `scripts/train_full.py` — the script is the single source of truth
(plan: `.claude/plans/titler-v2-8x4-d320-gated.md`).

Before running:
1. Google Drive `titler/` must contain: `train.jsonl`, `val.jsonl` (the 209k v1.1 set), the `tokenizer/` folder, **and `train_full.py`** (upload the current one from `scripts/`).
2. Runtime → Change runtime type → T4 GPU. Run all.
3. ~2.8× the v1 wall clock — expect an overnight run.
4. Download `titler-v2-8x4.zip` from Drive when done → unzip to `model/titler-v2-8x4/` in the repo.

**If Colab disconnects:** reconnect, set `RESUME = True` below, Run all. Checkpoints live on Drive and survive.

**Memory:** batch 128 OOMs even on a 24GB 4090 with this config (the SDPA position-bias masks are the hog), hence `BATCH_SIZE = 64` + gradient accumulation ×2 — same effective batch 128 as v1. If a T4 (16GB) still OOMs, drop to `BATCH_SIZE = 32` (accum becomes ×4 automatically).

In [ ]:
# ---- parameters ----
DRIVE_DIR = '/content/drive/MyDrive/titler'
OUT_NAME = 'titler-v2-8x4'
BATCH_SIZE = 64  # 128 OOMs on 24GB with this config; 64 + grad-accum keeps effective batch 128
RESUME = False  # set True after a disconnect to continue from the Drive checkpoint

ARCH = '--d-model 320 --num-heads 5 --d-ff 1024 --enc-layers 8 --dec-layers 4 --ff-proj gated-gelu'

In [ ]:
%pip install -q transformers datasets rouge-score accelerate
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU, then Run all again.'
from google.colab import drive
drive.mount('/content/drive')

import os
for f in ['train.jsonl', 'val.jsonl', 'tokenizer', 'train_full.py']:
    assert os.path.exists(f'{DRIVE_DIR}/{f}'), f'missing {DRIVE_DIR}/{f} — see the setup notes above'

In [ ]:
accum = max(1, 128 // BATCH_SIZE)  # keep effective batch 128 for v1-comparable dynamics
cmd = (f'python {DRIVE_DIR}/train_full.py --data-dir {DRIVE_DIR} --out {DRIVE_DIR}/{OUT_NAME} '
       f'--batch-size {BATCH_SIZE} --grad-accum {accum} {ARCH}' + (' --resume' if RESUME else ''))
print(cmd)
!{cmd}

In [ ]:
!cd {DRIVE_DIR} && zip -rq {OUT_NAME}.zip {OUT_NAME}
print('download', f'{DRIVE_DIR}/{OUT_NAME}.zip')